# 06 · Optionnel · Un feature store avec Feast

*Module 14 « Feature engineering & transformation » · TD du mercredi 21 octobre 2026 · notebook optionnel (≈ 25 min)*

**Ce qu'on fait.** Les variables d'historique « à date » de l'approche B (le passé de l'agent et du pays) ont besoin de tout le passé pour être calculées. On les range dans un *feature store* : on les calcule en SQL, on les enregistre dans un catalogue, puis on les sert **à l'identique** à l'entraînement (jointure à date) et en production (dernière valeur connue). Enfin, on regarde ce que le catalogue garde en mémoire pour tracer le tout.

**Les choix de cette approche**

| Décision | Choix | Pourquoi |
|---|---|---|
| Outil | Feast 0.67, en local (`provider: local`) | open source, s'installe avec `pip` ; les mêmes appels (`apply`, `get_historical_features`, `materialize`) servent avec BigQuery, Snowflake ou Redshift, en changeant la configuration et les sources |
| Entités (les clés) | `agent` (clé `agent`) et `pays` (clé `country`) | ce sont les deux historiques à date de l'approche B |
| Calcul des variables | SQL dans DuckDB, fonctions de fenêtre | comme dans un entrepôt ; Feast range et sert les variables, il ne les calcule pas |
| Horodatage | une « photo » par clé et par jour de changement, datée **jour + 1 jour** | la photo du jour J ne sert qu'aux réservations faites après J : le passé, rien que le passé |
| Magasin hors ligne | fichiers parquet (`type: file`) | pour l'entraînement : la jointure à date sur des milliers de réservations |
| Magasin en ligne | SQLite | pour la production : la dernière photo de chaque clé, lue en quelques millisecondes |
| Réservations sans agent ou sans pays | exclues de cette partie | une clé vide ne peut pas servir de clé d'entité ; le modèle B les traite avec `agent_absent` et l'imputation |
| Contrôle | comparaison avec `fe_tp.donnees.historique_a_date` | des valeurs identiques à l'entraînement et en production : pas de *training-serving skew* |

**Les outils utilisés**

- `duckdb` : `con.sql(...)` et les fonctions de fenêtre `SUM(...) OVER (PARTITION BY ... ORDER BY ...)` (cours 1, slide 14, « SQL dans l'entrepôt »).
- `feast` : `Entity`, `FileSource`, `FeatureView`, `Field`, `FeatureService` dans `definitions.py` ; `FeatureStore.apply` pour le catalogue (cours 1, slide 14, « Feature store »).
- `FeatureStore.get_historical_features` : la jointure à date, l'équivalent Feast de `pd.merge_asof` (cours 1, slide 14).
- `FeatureStore.materialize` et `get_online_features` : servir en production les valeurs vues à l'entraînement (cours 2, slide 24, *training-serving skew*).
- `FeatureStore.create_saved_dataset`, `list_feature_views`, `feast ui` : tracer.
- `fe_tp.donnees.historique_a_date` : la référence, calculée en pandas (cours 1, slides 13 et 14).

**Ce qu'on regarde à la fin.** Un tableau où l'écart maximal entre les valeurs servies par Feast et celles du paquet `fe_tp` vaut exactement 0, avec des valeurs manquantes aux mêmes endroits ; puis les mêmes valeurs, servies en ligne le jour de la mise en service.

## 1. Pourquoi un feature store ?

Le taux d'annulation passé d'un agent ne se lit pas dans la réservation : il se calcule sur **tout l'historique**, à la date de la réservation. Deux usages, deux contraintes :

- **à l'entraînement**, il faut pour chaque réservation passée la valeur *telle qu'on la connaissait ce jour-là* (cours 1, slide 14) ;
- **en production**, il faut la dernière valeur connue, tout de suite, pour une réservation qui arrive seule.

Si l'équipe qui entraîne et l'équipe qui sert recodent chacune le calcul, les valeurs finissent par diverger : c'est le *training-serving skew* (cours 2, slide 24). Le feature store est un **entrepôt de variables datées**, servies à l'identique des deux côtés, avec un **catalogue** partagé.

| Mot de Feast | Ici | Rôle |
|---|---|---|
| entité (*entity*) | `agent`, `pays` | la clé sur laquelle on joint les variables |
| source (*data source*) | `data/historique_agent.parquet` | l'historique des photos datées |
| vue (*feature view*) | `historique_agent` | des variables, pour une entité, depuis une source, avec une durée de validité (TTL) |
| service (*feature service*) | `modele_annulation_v1` | l'ensemble des variables qu'un modèle consomme, sous un nom versionné |
| registre (*registry*) | `data/registry.db` | le catalogue : définitions, dates, matérialisations |
| magasin hors ligne / en ligne | parquet / SQLite | tout l'historique (entraînement) / la dernière photo (production) |

```text
definitions.py ──apply──▶ registre (le catalogue : entités, vues, service)

hotels.csv ──SQL (DuckDB)──▶ photos datées (parquet = magasin hors ligne)
                                   │
                                   ├── get_historical_features ──▶ jeu d'entraînement (jointure à date)
                                   │
                                   └── materialize ──▶ magasin en ligne (SQLite) ──▶ get_online_features (production)
```

## 2. Le dépôt Feast

On importe les outils. Feast écrit beaucoup de journaux : on règle son *logger* au niveau `WARNING` (les avertissements et les erreurs restent visibles). On fait taire un seul message, précis : l'avertissement qui rappelle que l'enregistrement de jeux de données (section 7) est une fonction expérimentale de Feast. Enfin, on élargit l'affichage des colonnes de pandas pour lire les descriptions du catalogue en entier.

In [1]:
import contextlib
import io
import logging
import re
import runpy
import shutil
import warnings
from datetime import datetime
from pathlib import Path

import duckdb
import pandas as pd
from feast import FeatureStore
from feast.infra.offline_stores.file_source import SavedDatasetFileStorage

from fe_tp.donnees import charger, decouper, historique_a_date

logging.getLogger("feast").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="Saving dataset is an experimental feature", category=RuntimeWarning)
pd.set_option("display.max_colwidth", 100)

Le dépôt est le dossier `feature_store/` à la racine du projet. On vide son sous-dossier `data/` pour repartir de zéro : tout ce qu'il contient est régénéré par ce notebook (et ignoré par Git).

In [2]:
DEPOT = Path("../feature_store")
shutil.rmtree(DEPOT / "data", ignore_errors=True)
(DEPOT / "data").mkdir()
sorted(chemin.name for chemin in DEPOT.iterdir() if chemin.name != "__pycache__")

['README.md', 'data', 'definitions.py', 'feature_store.yaml']

Le dépôt contient la configuration (`feature_store.yaml`), les définitions (`definitions.py`), un court `README.md` et le dossier `data/`, vide pour l'instant.

La configuration tient en quelques lignes : le nom du projet, l'emplacement du registre et les deux magasins.

In [3]:
print((DEPOT / "feature_store.yaml").read_text())

# Le dépôt Feast du TD (notebook 06). Les chemins sont relatifs à ce dossier.
project: hotel_annulations
provider: local
registry: data/registry.db          # le catalogue : entités, vues, services, dates
online_store:                       # le magasin en ligne : la dernière photo de chaque clé
  type: sqlite
  path: data/online_store.db
offline_store:                      # le magasin hors ligne : l'historique des photos (parquet)
  type: file
entity_key_serialization_version: 3



Les définitions sont **du code** : un fichier Python qui se relit, se discute en revue de code et se versionne avec Git, comme le reste du projet.

In [4]:
print((DEPOT / "definitions.py").read_text())

"""Les définitions du feature store du TD : deux entités, deux vues de variables, un service.

Ce fichier est du code : il se relit, se versionne avec Git et s'enregistre dans le registre
avec `feast apply` (ou `FeatureStore.apply` dans le notebook 06).

Chaque ligne des fichiers parquet est une « photo » : ce qu'on savait d'un agent (ou d'un pays)
à l'instant `date_photo`. La photo du jour J est horodatée J + 1 jour : elle ne sert qu'aux
réservations faites après J, comme `fe_tp.donnees.historique_a_date`.
"""

from datetime import timedelta

from feast import Entity, FeatureService, FeatureView, Field, FileSource, ValueType
from feast.types import Float64, Int64, String

PROPRIETAIRE = "AMA · module 14"
DIX_ANS = timedelta(days=3650)  # une photo reste valable longtemps : les données vont de 2013 à 2017

# Les entités : les clés sur lesquelles on joint les variables
agent = Entity(
    name="agent",
    join_keys=["agent"],
    value_type=ValueType.STRING,
    description="Agence de 

Le fichier déclare sept objets Feast : deux entités (`agent` et `pays`), deux sources parquet horodatées par `date_photo`, deux vues de trois variables chacune, avec une durée de validité (TTL) de 3 650 jours, et un service, `modele_annulation_v1`, qui réunit les deux vues. On n'en passera que cinq à `apply` (section 4) : chaque source est enregistrée avec sa vue. Les variables portent les noms du paquet `fe_tp` (`taux_annul_passe_agent`…) ; les descriptions et les tags, en français, deviennent la documentation du catalogue.

## 3. Calculer les variables dans l'entrepôt, en SQL

Feast ne calcule pas les variables : il les range et les sert. Le calcul se fait en amont, ici en SQL avec **DuckDB**, une base analytique qui tourne dans le processus Python, comme on le ferait dans l'entrepôt de l'entreprise (cours 1, slide 14). On charge les réservations et on les expose comme une table `reservations`.

In [5]:
df = charger()
train, test = decouper(df)
con = duckdb.connect()
con.register("reservations", df[["agent", "country", "date_reservation", "date_cloture", "is_canceled"]])
con.sql("SELECT COUNT(*) AS reservations, COUNT(agent) AS avec_agent, COUNT(country) AS avec_pays FROM reservations").df()

,reservations,avec_agent,avec_pays
0,119208,102928,118730


Sur 119 208 réservations, 102 928 ont un agent et 118 730 un pays : `COUNT(colonne)` ne compte que les valeurs renseignées. Les réservations sans agent n'entreront dans aucune photo d'agent.

**Une photo par clé et par jour de changement.** Pour chaque agent, deux sortes d'événements modifient ce qu'on sait de lui : une réservation **faite** (à sa date de réservation) et une réservation **close** (à sa date de clôture : séjour, annulation ou absence). La requête :

1. empile ces événements (`UNION ALL`) et les compte par clé et par jour (`GROUP BY`) ;
2. cumule les comptes avec une fonction de fenêtre, de la toute première ligne jusqu'au jour courant (`ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`) ;
3. date la photo du **lendemain** : `jour + INTERVAL 1 DAY`. La photo qui compte les événements du jour J ne servira qu'aux réservations faites à partir de J + 1, comme `historique_a_date`, qui ne garde que les issues closes **strictement avant** le jour de la réservation.

La clé (`agent` ou `country`) est un paramètre : la même requête sert pour les deux entités.

In [6]:
REQUETE = '''
WITH evenements AS (
    SELECT {cle}, date_reservation AS jour, 1 AS resa, 0 AS issue, 0 AS annulee FROM reservations
    UNION ALL
    SELECT {cle}, date_cloture, 0, 1, is_canceled FROM reservations
),
par_jour AS (
    SELECT {cle}, jour, SUM(resa) AS resa, SUM(issue) AS issues, SUM(annulee) AS annulees
    FROM evenements WHERE {cle} IS NOT NULL GROUP BY {cle}, jour
)
SELECT {cle}, jour + INTERVAL 1 DAY AS date_photo,
       CAST(SUM(resa) OVER w AS BIGINT) AS nb_resa_passees_{cle},
       CAST(SUM(issues) OVER w AS BIGINT) AS nb_issues_connues_{cle},
       CAST(SUM(annulees) OVER w AS DOUBLE) / NULLIF(SUM(issues) OVER w, 0) AS taux_annul_passe_{cle}
FROM par_jour
WINDOW w AS (PARTITION BY {cle} ORDER BY jour ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
ORDER BY {cle}, date_photo
'''

On exécute la requête pour les deux clés. Pour lire le résultat, on suit un petit agent, l'agent `11`, à ses débuts : d'abord ses réservations brutes, puis ses photos.

In [7]:
photos_agent = con.sql(REQUETE.format(cle="agent")).df()
photos_pays = con.sql(REQUETE.format(cle="country")).df()
print(len(photos_agent), "photos d'agents ;", len(photos_pays), "photos de pays")
df[df["agent"] == "11"].sort_values("date_reservation")[["reservation_id", "date_reservation", "date_cloture", "is_canceled"]].head(8)

22351 photos d'agents ; 22683 photos de pays


,reservation_id,date_reservation,date_cloture,is_canceled
486,76155,2015-01-02,2015-07-07,0
6025,77015,2015-03-02,2015-08-29,0
537,40117,2015-03-12,2015-04-08,1
1748,40297,2015-04-06,2015-05-25,1
2655,40552,2015-04-06,2015-05-05,1
5311,41598,2015-04-06,2015-05-05,1
535,40115,2015-04-08,2015-04-08,1
536,40116,2015-04-08,2015-04-11,1


In [8]:
photos_agent[photos_agent["agent"] == "11"].head(6)

,agent,date_photo,nb_resa_passees_agent,nb_issues_connues_agent,taux_annul_passe_agent
614,11,2015-01-03,1,0,NaN
615,11,2015-03-03,2,0,NaN
616,11,2015-03-13,3,0,NaN
617,11,2015-04-07,6,0,NaN
618,11,2015-04-09,8,2,1.0
619,11,2015-04-12,8,3,1.0


La requête produit 22 351 photos d'agents et 22 683 photos de pays : une ligne par clé et par jour de changement, pas une par réservation. L'agent `11` fait sa première réservation le 2 janvier 2015 : la photo datée du 3 janvier compte 1 réservation passée, aucune issue connue, et un taux vide. Le 8 avril, deux réservations sont closes par une annulation (40117 et 40115) et deux nouvelles sont saisies : la photo datée du 9 avril passe à 8 réservations, 2 issues connues et un taux de 1,0. Une réservation de l'agent `11` saisie le 8 avril voit encore la photo du 7 avril (6 réservations, aucune issue connue) : les annulations du jour même ne comptent pas encore.

> 💬 **Pour la salle.** Pourquoi dater la photo du jour J au **lendemain**, et pas le jour même ?
>
> *Piste : le fichier ne donne que des jours, pas des heures. Une réservation saisie le 8 avril au matin ne pouvait pas connaître une annulation close le 8 avril au soir. Dater la photo du 8 avril ferait entrer ces issues dans les variables : une fuite par le futur, petite mais réelle (cours 1, slide 14, la frise A/B/C).*

On écrit les photos dans les fichiers parquet que `definitions.py` déclare comme sources. Feast travaille en temps universel : on le dit explicitement avec `tz_localize("UTC")`.

In [9]:
for nom, photos in [("agent", photos_agent), ("pays", photos_pays)]:
    photos["date_photo"] = photos["date_photo"].dt.tz_localize("UTC")
    photos.to_parquet(DEPOT / "data" / f"historique_{nom}.parquet", index=False)
sorted(chemin.name for chemin in (DEPOT / "data").iterdir())

['historique_agent.parquet', 'historique_pays.parquet']

## 4. Enregistrer les définitions : le catalogue

On ouvre le dépôt avec `FeatureStore`, on exécute `definitions.py` (avec `runpy`, comme le ferait `feast apply`) et on enregistre ses objets dans le registre avec `apply`.

In [10]:
store = FeatureStore(repo_path=str(DEPOT))
definitions = runpy.run_path(str(DEPOT / "definitions.py"))
objets = [definitions[nom] for nom in ["agent", "pays", "historique_agent", "historique_pays", "modele_annulation_v1"]]
store.apply(objets)
print("projet :", store.project)

projet : hotel_annulations


Le registre est maintenant un **catalogue** que toute l'équipe peut interroger : quelles variables existent, pour quelle entité, d'où elles viennent, combien de temps elles restent valables.

In [11]:
catalogue = pd.DataFrame([
    {"vue": vue.name, "entité": vue.entities[0], "variable": champ.name, "type": str(champ.dtype),
     "description": champ.description}
    for vue in store.list_feature_views() for champ in vue.features
])
catalogue

,vue,entité,variable,type,description
0,historique_agent,agent,nb_resa_passees_agent,Int64,Réservations faites avant le jour de la réservation.
1,historique_agent,agent,nb_issues_connues_agent,Int64,"Réservations closes (séjour, annulation ou absence) avant ce jour."
2,historique_agent,agent,taux_annul_passe_agent,Float64,Part d'annulations parmi les issues connues ; vide s'il n'y en a aucune.
3,historique_pays,pays,nb_resa_passees_country,Int64,Réservations faites avant le jour de la réservation.
4,historique_pays,pays,nb_issues_connues_country,Int64,"Réservations closes (séjour, annulation ou absence) avant ce jour."
5,historique_pays,pays,taux_annul_passe_country,Float64,Part d'annulations parmi les issues connues ; vide s'il n'y en a aucune.


In [12]:
for vue in store.list_feature_views():
    print(f"vue {vue.name} · source {vue.batch_source.path} · TTL {vue.ttl.days} jours · tags {vue.tags}")

vue historique_agent · source data/historique_agent.parquet · TTL 3650 jours · tags {'fuite': 'aucune', 'calcul': 'SQL (fonctions de fenêtre)', 'cours': 'module 14, cours 1, slide 14'}
vue historique_pays · source data/historique_pays.parquet · TTL 3650 jours · tags {'fuite': 'aucune', 'calcul': 'SQL (fonctions de fenêtre)', 'cours': 'module 14, cours 1, slide 14'}


In [13]:
for entite in store.list_entities():
    print(f"entité {entite.name!r} · clé {entite.join_key!r} · {entite.description}")
service = store.get_feature_service("modele_annulation_v1")
for projection in service.feature_view_projections:
    print(f"service {service.name} ← {projection.name} : {[f.name for f in projection.features]}")

entité 'agent' · clé 'agent' · Agence de voyages qui a saisi la réservation (identifiant anonymisé du fichier).
entité 'pays' · clé 'country' · Pays d'origine du client : code ISO 3166-1 à trois lettres, sauf « CN », doublon de « CHN », et « TMP », ancien code du Timor oriental.
service modele_annulation_v1 ← historique_agent : ['nb_resa_passees_agent', 'nb_issues_connues_agent', 'taux_annul_passe_agent']
service modele_annulation_v1 ← historique_pays : ['nb_resa_passees_country', 'nb_issues_connues_country', 'taux_annul_passe_country']


Le catalogue liste six variables, trois par vue, chacune avec son type (`Int64` pour les comptes, `Float64` pour les taux) et sa description. Chaque vue indique sa source parquet, son TTL de 3 650 jours et ses tags (la slide du cours, la méthode de calcul). Le service `modele_annulation_v1` réunit les deux vues : c'est sous ce nom que le modèle demandera ses variables, à l'entraînement comme en production.

En ligne de commande, depuis le dossier `feature_store/`, les mêmes opérations s'écrivent :

```bash
uv run feast apply                       # enregistrer les définitions (lit definitions.py)
uv run feast entities list               # les entités
uv run feast feature-views list          # les vues de variables
uv run feast feature-services list       # les services
uv run feast ui                          # parcourir le catalogue dans le navigateur
```

## 5. Entraînement : la jointure à date

`get_historical_features` reçoit un tableau d'entités (*entity dataframe*) : une clé et un horodatage `event_timestamp` par ligne. Pour chaque ligne, Feast cherche la **dernière photo antérieure** (photo ≤ horodatage, et pas plus vieille que le TTL). C'est le `pd.merge_asof` de la slide 14, fait par le feature store.

On demande au catalogue la liste des variables du service `modele_annulation_v1`, vue par vue.

In [14]:
variables = {projection.name: [f"{projection.name}:{f.name}" for f in projection.features]
             for projection in service.feature_view_projections}
variables

{'historique_agent': ['historique_agent:nb_resa_passees_agent',
  'historique_agent:nb_issues_connues_agent',
  'historique_agent:taux_annul_passe_agent'],
 'historique_pays': ['historique_pays:nb_resa_passees_country',
  'historique_pays:nb_issues_connues_country',
  'historique_pays:taux_annul_passe_country']}

Le tableau d'entités, ce sont les réservations du **train** qui ont un agent, horodatées à leur date de réservation. Les réservations sans agent sont écartées : une clé vide ne peut pas être une clé d'entité. Deux précautions, propres au magasin hors ligne local de Feast :

- il ne garde qu'une ligne par couple (clé, horodatage) : on lui envoie donc les couples **distincts** (agent, jour), puis on refusionne sur toutes les réservations ;
- il joint chaque ligne à toutes les photos de sa clé avant de filtrer : envoyer les couples distincts allège aussi beaucoup le calcul.

In [15]:
def entites(reservations, cle):
    # Les réservations qui ont une clé, horodatées en temps universel comme le veut Feast.
    avec_cle = reservations.dropna(subset=[cle])
    return avec_cle.assign(event_timestamp=avec_cle["date_reservation"].dt.tz_localize("UTC"))


resa_agent = entites(train, "agent")
entites_agent = resa_agent[["agent", "event_timestamp"]].drop_duplicates()
print(len(train) - len(resa_agent), "réservations du train sans agent, écartées")
print(len(resa_agent), "réservations avec agent →", len(entites_agent), "couples (agent, jour) distincts")

13730 réservations du train sans agent, écartées
81460 réservations avec agent → 11495 couples (agent, jour) distincts


Les 13 730 réservations du train sans agent sortent de cette partie (dans l'approche B, le modèle les traite avec `agent_absent` et l'imputation). Les 81 460 autres se résument à 11 495 couples (agent, jour) distincts : on envoie sept fois moins de lignes à Feast.

In [16]:
job_agent = store.get_historical_features(entity_df=entites_agent, features=variables["historique_agent"])
servi_agent = job_agent.to_df()
print(len(entites_agent), "lignes demandées,", len(servi_agent), "lignes servies")

11495 lignes demandées, 11188 lignes servies


Feast renvoie 11 188 lignes au lieu de 11 495 : 307 couples ont disparu, sans erreur ni avertissement.

**Le piège : des lignes perdues.** Lesquelles ? On compare chaque couple à la toute première photo de son agent.

In [17]:
premiere_photo = photos_agent.groupby("agent")["date_photo"].min()
sans_photo = entites_agent["event_timestamp"] < entites_agent["agent"].map(premiere_photo)
perdus = entites_agent.merge(servi_agent, how="left", indicator=True)["_merge"] == "left_only"
print(sans_photo.sum(), "couples (agent, jour) n'ont aucune photo antérieure ;",
      "ce sont exactement les couples perdus :", (sans_photo.to_numpy() == perdus.to_numpy()).all())
concernees = resa_agent.merge(entites_agent[sans_photo], on=["agent", "event_timestamp"])
print(len(concernees), "réservations concernées ; taux d'annulation :", round(concernees["is_canceled"].mean(), 3),
      "contre", round(resa_agent["is_canceled"].mean(), 3), "sur l'ensemble")

307 couples (agent, jour) n'ont aucune photo antérieure ; ce sont exactement les couples perdus : True
5758 réservations concernées ; taux d'annulation : 0.622 contre 0.385 sur l'ensemble


Les 307 couples perdus sont exactement ceux qui n'ont aucune photo antérieure : le tout premier jour de réservation d'un agent, avant lequel on ne savait rien de lui. Ils pèsent 5 758 réservations (en moyenne 19 par couple : des blocs saisis le même jour), annulées à 62 % contre 38,5 % sur l'ensemble. Les perdre, c'est retirer du train le cas le plus risqué : le nouvel agent. (Autre parade possible : ajouter à chaque clé une photo « zéro », datée de son premier jour.)

On refusionne donc sur **toutes** les réservations, avec une jointure à gauche. Une réservation sans photo antérieure reçoit la convention de `historique_a_date` : 0 réservation passée, 0 issue connue, taux vide (rien n'était connu).

In [18]:
def refusionner(reservations, servi, cle):
    # Chaque réservation reçoit la photo de son couple (clé, jour) ; sans photo, rien n'était connu.
    tout = reservations[["reservation_id", cle, "event_timestamp"]].merge(servi, on=[cle, "event_timestamp"], how="left")
    comptes = [f"nb_resa_passees_{cle}", f"nb_issues_connues_{cle}"]
    tout[comptes] = tout[comptes].fillna(0)
    return tout.set_index("reservation_id")


agent_feast = refusionner(resa_agent, servi_agent, "agent")
print(len(agent_feast), "réservations avec leurs variables d'agent")

81460 réservations avec leurs variables d'agent


Même chose pour le pays, en une cellule.

In [19]:
resa_pays = entites(train, "country")
entites_pays = resa_pays[["country", "event_timestamp"]].drop_duplicates()
job_pays = store.get_historical_features(entity_df=entites_pays, features=variables["historique_pays"])
servi_pays = job_pays.to_df()
pays_feast = refusionner(resa_pays, servi_pays, "country")
print(len(train) - len(resa_pays), "réservations sans pays écartées ;", len(entites_pays), "couples demandés,",
      len(servi_pays), "servis ;", len(pays_feast), "réservations refusionnées")

473 réservations sans pays écartées ; 13281 couples demandés, 13117 servis ; 94717 réservations refusionnées


Pour le pays, 473 réservations sans pays sont écartées ; Feast sert 13 117 couples sur 13 281 demandés (164 perdus, pour la même raison), et les 94 717 réservations retrouvent toutes leurs variables après la refusion.

**Le contrôle.** On calcule les mêmes variables avec `historique_a_date` (pandas, `merge_asof`) et on compare, colonne par colonne, **sans aucune tolérance**.

In [20]:
reference = pd.concat([historique_a_date(df, "agent"), historique_a_date(df, "country")], axis=1)
reference.index = df["reservation_id"]
lignes = []
for cle, feast_df in [("agent", agent_feast), ("country", pays_feast)]:
    for colonne in [f"nb_resa_passees_{cle}", f"nb_issues_connues_{cle}", f"taux_annul_passe_{cle}"]:
        a, b = feast_df[colonne], reference.loc[feast_df.index, colonne]
        lignes.append({"variable": colonne, "réservations": len(a), "vides (Feast)": a.isna().sum(),
                       "vides (fe_tp)": b.isna().sum(), "vides aux mêmes endroits": a.isna().equals(b.isna()),
                       "écart maximal": (a - b).abs().max()})
pd.DataFrame(lignes)

,variable,réservations,vides (Feast),vides (fe_tp),vides aux mêmes endroits,écart maximal
0,nb_resa_passees_agent,81460,0,0,True,0.0
1,nb_issues_connues_agent,81460,0,0,True,0.0
2,taux_annul_passe_agent,81460,8818,8818,True,0.0
3,nb_resa_passees_country,94717,0,0,True,0.0
4,nb_issues_connues_country,94717,0,0,True,0.0
5,taux_annul_passe_country,94717,4797,4797,True,0.0


In [21]:
for cle, feast_df in [("agent", agent_feast), ("country", pays_feast)]:
    colonnes = [f"nb_resa_passees_{cle}", f"nb_issues_connues_{cle}", f"taux_annul_passe_{cle}"]
    pd.testing.assert_frame_equal(feast_df[colonnes], reference.loc[feast_df.index, colonnes], check_exact=True)
print("Identiques, au bit près.")

Identiques, au bit près.


Sur les 81 460 réservations avec agent et les 94 717 avec pays, l'écart maximal vaut 0,0 pour les six variables, et les valeurs vides tombent aux mêmes endroits (8 818 taux d'agent et 4 797 taux de pays vides : aucune issue connue à cette date). `assert_frame_equal(..., check_exact=True)` le confirme sans aucune tolérance : SQL et Feast d'un côté, pandas et `merge_asof` de l'autre, donnent exactement les mêmes valeurs.

> 💬 **Pour la salle.** Si l'on n'avait pas vérifié le nombre de lignes, quel modèle aurait-on entraîné ?
>
> *Piste : un modèle qui n'a jamais vu la première réservation d'un agent. Or en production, de nouveaux agents arrivent : le modèle découvrirait ce cas le jour J. Une jointure qui perd des lignes en silence biaise le jeu d'entraînement ; on compte toujours les lignes avant et après une jointure.*

## 6. Production : le magasin en ligne

En production, une réservation arrive seule : on n'a pas le temps de rejouer une jointure à date sur des parquets. On **matérialise** : Feast copie la dernière photo de chaque clé dans le magasin en ligne (SQLite), où elle se lit en quelques millisecondes.

Le modèle est mis en service le **22 avril 2017** (la date de coupure du train). On matérialise donc les photos jusqu'au 22 avril 2017 à 0 h, avec des dates **explicites**. `materialize_incremental` partirait, au premier appel, de « maintenant − TTL » : nous sommes en 2026, elle commencerait en octobre 2016 et laisserait de côté les clés dont la dernière photo est plus ancienne. Feast imprime son journal en couleurs (codes ANSI) : on le récupère et on l'affiche en texte simple.

In [22]:
journal = io.StringIO()
with contextlib.redirect_stdout(journal):
    store.materialize(start_date=datetime(2013, 6, 1), end_date=datetime(2017, 4, 22))
print(re.sub(r"\x1b\[[0-9;]*m", "", journal.getvalue()).strip())

Materializing 2 feature views from 2013-06-01 00:00:00+00:00 to 2017-04-22 00:00:00+00:00 into the sqlite online store.

historique_agent:
historique_pays:


Le 22 avril 2017, des réservations arrivent. On prend quatre réservations saisies ce jour-là (quatre agents différents), plus une réservation fictive avec un agent et un pays jamais vus (`"999"`, `"XKX"` pour le Kosovo). On interroge le magasin en ligne avec **le même service** que pour l'entraînement.

In [23]:
du_jour = df[df["date_reservation"] == "2017-04-22"].dropna(subset=["agent", "country"]).drop_duplicates("agent").head(4)
demandes = du_jour[["agent", "country"]].to_dict("records") + [{"agent": "999", "country": "XKX"}]
en_ligne = store.get_online_features(features=service, entity_rows=demandes).to_df()
en_ligne[["agent", "country", "nb_resa_passees_agent", "taux_annul_passe_agent", "nb_resa_passees_country", "taux_annul_passe_country"]]

,agent,country,nb_resa_passees_agent,taux_annul_passe_agent,nb_resa_passees_country,taux_annul_passe_country
0,527,SWE,34.0,NaN,924.0,0.248031
1,9,AUT,29491.0,0.451946,1190.0,0.214571
2,241,GBR,1456.0,0.145136,11393.0,0.224638
3,240,ESP,12834.0,0.411244,7777.0,0.251162
4,999,XKX,NaN,NaN,NaN,NaN


Le magasin en ligne répond pour les quatre agents connus. L'agent `9` avait fait 29 491 réservations avant le 22 avril 2017, avec un taux d'annulation connu de 0,452 ; l'agent `527` en avait fait 34, mais aucune n'était close : son taux est vide (aucune issue connue), comme à l'entraînement dans la même situation. Pour l'agent `999` et le pays `XKX`, jamais vus, toutes les valeurs sont vides, comptes compris : à l'entraînement, ce cas recevait 0 réservation passée (voir la question plus bas).

Ces valeurs sont-elles celles qu'aurait calculées `historique_a_date` pour ces mêmes réservations ? On compare les six variables, ligne à ligne.

In [24]:
colonnes = list(reference.columns)
servies = en_ligne.head(4)[colonnes].astype(float).set_axis(du_jour["reservation_id"])
attendues = reference.loc[du_jour["reservation_id"], colonnes]
print("écart maximal :", (servies - attendues).abs().max().max())
print("identiques, vides compris :", servies.equals(attendues))

écart maximal : 0.0
identiques, vides compris : True


Écart maximal de 0,0, et les valeurs vides aux mêmes endroits : en production, le modèle reçoit exactement ce que `historique_a_date` (donc l'entraînement) aurait calculé pour une réservation saisie ce jour-là. Pas de *training-serving skew*.

Le lendemain, on rafraîchit le magasin en ligne avec les photos suivantes. Dans la vraie vie, cette commande tourne chaque nuit, planifiée (cron, Airflow…). Ici, on rattrape d'un coup la fin de l'historique, jusqu'au 1er octobre 2017.

In [25]:
with contextlib.redirect_stdout(io.StringIO()):
    store.materialize(start_date=datetime(2017, 4, 22), end_date=datetime(2017, 10, 1))
apres = store.get_online_features(features=service, entity_rows=demandes[:2]).to_df()
apres[["agent", "country", "nb_resa_passees_agent", "nb_issues_connues_agent", "taux_annul_passe_agent"]]

,agent,country,nb_resa_passees_agent,nb_issues_connues_agent,taux_annul_passe_agent
0,527,SWE,35,35,0.00000
1,9,AUT,31922,31922,0.41545


Après le rafraîchissement, l'agent `9` compte 31 922 réservations passées et un taux de 0,415 (contre 29 491 et 0,452 le 22 avril). L'agent `527` a maintenant 35 réservations passées, toutes closes et aucune annulée : son taux passe de vide à 0,0. Le magasin en ligne ne garde que la dernière photo de chaque clé ; tout l'historique reste dans le magasin hors ligne.

> 💬 **Pour la salle.** L'agent `"999"` n'existe pas dans le magasin : Feast renvoie des valeurs vides. Le modèle va-t-il planter ?
>
> *Piste : non, le pipeline B impute les valeurs numériques vides (`SimpleImputer`). Le vrai risque est un décalage silencieux. À l'entraînement, un agent sans passé recevait 0 réservation passée, 0 issue connue et un taux vide (la règle de `refusionner`) : en production, il faut lui appliquer la même convention, et seul le taux vide passe par l'imputation. Si l'on confiait aussi les comptes vides à l'imputation, le nouvel agent recevrait les comptes médians du train, comme un agent déjà installé : un cas que le modèle n'a jamais vu à l'entraînement. Et le cas « agent sans passé » existe bien à l'entraînement : ce sont les lignes qu'on a failli perdre en section 5.*

## 7. Tracer : ce que le registre sait

Un feature store n'est pas qu'un cache : son registre garde **qui** a défini **quoi**, **quand**, à partir de **quelle source**, et ce qui a été matérialisé.

In [26]:
pd.DataFrame([
    {"vue": vue.name, "créée le (UTC)": vue.created_timestamp.replace(microsecond=0),
     "modifiée le (UTC)": vue.last_updated_timestamp.replace(microsecond=0), "source": vue.batch_source.path,
     "propriétaire": vue.owner,
     "matérialisée": " ; ".join(f"{debut:%Y-%m-%d} → {fin:%Y-%m-%d}" for debut, fin in vue.materialization_intervals)}
    for vue in store.list_feature_views()
])

,vue,créée le (UTC),modifiée le (UTC),source,propriétaire,matérialisée
0,historique_agent,2026-10-21 10:30:38,2026-10-21 10:30:48,data/historique_agent.parquet,AMA · module 14,2013-06-01 → 2017-04-22 ; 2017-04-22 → 2017-10-01
1,historique_pays,2026-10-21 10:30:38,2026-10-21 10:30:48,data/historique_pays.parquet,AMA · module 14,2013-06-01 → 2017-04-22 ; 2017-04-22 → 2017-10-01


Pour chaque vue, le registre a gardé sa date de création, sa dernière modification (chaque matérialisation la met à jour), sa source, son propriétaire et les deux intervalles matérialisés : du 1er juin 2013 au 22 avril 2017, puis du 22 avril au 1er octobre 2017. On sait donc, à tout moment, jusqu'où le magasin en ligne est à jour. (Les dates de création sont celles de l'exécution de ce notebook.)

**Enregistrer les valeurs servies.** Pour pouvoir dire plus tard « voici les valeurs que Feast a servies pour l'entraînement », on enregistre le résultat des deux jointures à date comme **jeux de données** (*saved datasets*) : Feast écrit les lignes dans un parquet et garde dans le registre les variables, les clés, la période et le service concerné. Une ligne est un couple (clé, jour), pas une réservation : la liste des réservations du train et la règle de `refusionner` restent dans le code.

In [27]:
for vue, job in [("historique_agent", job_agent), ("historique_pays", job_pays)]:
    store.create_saved_dataset(
        from_=job, name=f"entrainement_v1_{vue}", feature_service=service,
        storage=SavedDatasetFileStorage(path=f"data/entrainement_v1_{vue}.parquet"),
        tags={"modele": "approche B · gradient boosting", "periode": "train : arrivées avant le 2017-04-22"},
    )

In [28]:
pd.DataFrame([
    {"jeu": jeu.name, "service": jeu.feature_service_name, "clés": ", ".join(jeu.join_keys),
     "variables": len(jeu.features), "du": f"{jeu.min_event_timestamp:%Y-%m-%d}",
     "au": f"{jeu.max_event_timestamp:%Y-%m-%d}", "fichier": jeu.storage.file_options.uri}
    for jeu in store.list_saved_datasets()
])

,jeu,service,clés,variables,du,au,fichier
0,entrainement_v1_historique_agent,modele_annulation_v1,agent,3,2014-04-18,2017-04-21,data/entrainement_v1_historique_agent.parquet
1,entrainement_v1_historique_pays,modele_annulation_v1,country,3,2013-06-24,2017-04-21,data/entrainement_v1_historique_pays.parquet


Les deux jeux sont enregistrés avec leur service (`modele_annulation_v1`), leur clé, leurs trois variables, leur fichier et une période. Attention : cette période est celle du tableau d'entités **envoyé** à Feast (couples demandés du 18 avril 2014 au 21 avril 2017 pour les agents, du 24 juin 2013 au 21 avril 2017 pour les pays), pas celle des lignes du fichier. Le fichier ne contient que ce que Feast a servi : les couples sans photo antérieure, dont les tout premiers, n'y figurent pas. Leur convention (« rien n'était connu ») vit dans le code de `refusionner`, à versionner avec le reste.

Le reste de la traçabilité passe par les outils habituels :

- **Git** : `feature_store.yaml` et `definitions.py` sont versionnés ; `git log feature_store/definitions.py` dit qui a changé quelle définition, et pourquoi. Le dossier `data/` (registre, parquets, magasin en ligne) est régénéré, il est ignoré par Git.
- **Le nom du service porte la version** : si l'on change la formule d'une variable, on crée `modele_annulation_v2` au lieu de modifier `v1` en silence. Le modèle déjà en service continue de recevoir exactement ce qu'il a vu à l'entraînement.
- **`feast ui`** (depuis `feature_store/`) ouvre le catalogue dans le navigateur : entités, vues, services, jeux enregistrés, et le lignage des sources vers les services.

| Question | Où se trouve la réponse |
|---|---|
| D'où vient `taux_annul_passe_agent` ? | vue `historique_agent` → source `data/historique_agent.parquet` → requête SQL de la section 3 → `hotels.csv` |
| Quelles variables consomme le modèle ? | service `modele_annulation_v1` |
| Quelles valeurs ont servi à l'entraînement ? | jeux `entrainement_v1_historique_agent` et `entrainement_v1_historique_pays` (une ligne par couple (clé, jour) servi) ; `refusionner` les rattache aux réservations du train |
| Jusqu'où le magasin en ligne est-il à jour ? | intervalles de matérialisation de chaque vue |
| Qui a changé une définition, et quand ? | `git log`, et les dates du registre |

> 💬 **Pour la salle.** Un collègue veut ne compter, dans le taux d'annulation passé, que les deux dernières années. Que doit-il faire, et que ne doit-il surtout pas faire ?
>
> *Piste : écrire une nouvelle requête et une nouvelle vue (ou de nouvelles variables), les exposer dans un service `modele_annulation_v2`, entraîner un nouveau modèle dessus, et comparer. Ne surtout pas modifier la vue existante en place : le modèle v1, en production, recevrait des valeurs qu'il n'a jamais vues à l'entraînement.*

## 8. À retenir

- Un feature store est un **entrepôt de variables datées** : il les sert à l'identique à l'entraînement (jointure à date) et en production (dernière valeur), ce qui supprime le *training-serving skew*.
- **Le passé, rien que le passé** se règle à l'horodatage : la photo du jour J est datée J + 1. Avec cette règle, Feast et `historique_a_date` donnent ici les mêmes valeurs, au bit près.
- Le calcul reste du **SQL dans l'entrepôt** (fonctions de fenêtre) ; Feast enregistre, joint à date, matérialise et sert.
- **Comptez les lignes** avant et après chaque jointure : la jointure à date du magasin hors ligne local de Feast a laissé tomber, en silence, les couples sans photo antérieure. Côté agents, ce sont 5 758 réservations faites le premier jour d'un agent, annulées à 62 % contre 38,5 %.
- Les définitions sont **du code versionné**, et les services portent une version : on trace qui a défini quoi, quand, à partir de quoi, et quelles valeurs ont été servies pour l'entraînement.